Este proyecto se centra en la premisa: “Cliente X tiene alta probabilidad de pago si se contacta en canal Y y momento Z”

In [2]:
pip install kaggle

Ajustes preliminares

In [7]:
from google.colab import files
files.upload()  # subes kaggle.json

Saving kaggle.json to kaggle.json


{'kaggle.json': b'{"username":"eusebi0","key":"f8b6277736da7a02740fe32f907fd21b"}'}

In [1]:
pip install pandas numpy scikit-learn xgboost imbalanced-learn shap matplotlib seaborn streamlit joblib plotly

Se inicia descargando el dataset: "Give me some credit" de Kaggle

In [14]:
# ============================================================
# SISTEMA DE COBRANZA INTELIGENTE CON IA
# ============================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, roc_auc_score, confusion_matrix
import warnings
warnings.filterwarnings('ignore')

# --- CARGA DE DATOS ---
df = pd.read_csv('cs-training.csv', index_col=0)

print("Shape del dataset:", df.shape)
print("\nPrimeras filas:")
df.head()

Shape del dataset: (150000, 11)

Primeras filas:


,SeriousDlqin2yrs,RevolvingUtilizationOfUnsecuredLines,age,NumberOfTime30-59DaysPastDueNotWorse,DebtRatio,MonthlyIncome,NumberOfOpenCreditLinesAndLoans,NumberOfTimes90DaysLate,NumberRealEstateLoansOrLines,NumberOfTime60-89DaysPastDueNotWorse,NumberOfDependents
1,1,0.766127,45,2,0.802982,9120.0,13,0,6,0,2.0
2,0,0.957151,40,0,0.121876,2600.0,4,0,0,0,1.0
3,0,0.658180,38,1,0.085113,3042.0,2,1,0,0,0.0
4,0,0.233810,30,0,0.036050,3300.0,5,0,0,0,0.0
5,0,0.907239,49,1,0.024926,63588.0,7,0,1,0,0.0


Se procede a hacer una exploración general del dataset.

In [15]:
# --- EXPLORACIÓN BÁSICA ---
print("=== RESUMEN DEL DATASET ===")
print(df.info())
print("\n=== ESTADÍSTICAS ===")
print(df.describe())
print("\n=== VALORES NULOS ===")
print(df.isnull().sum())
print("\n=== TASA DE MOROSIDAD ===")
print(df['SeriousDlqin2yrs'].value_counts(normalize=True))

=== RESUMEN DEL DATASET ===
<class 'pandas.core.frame.DataFrame'>
Index: 150000 entries, 1 to 150000
Data columns (total 11 columns):
 #   Column                                Non-Null Count   Dtype  
---  ------                                --------------   -----  
 0   SeriousDlqin2yrs                      150000 non-null  int64  
 1   RevolvingUtilizationOfUnsecuredLines  150000 non-null  float64
 2   age                                   150000 non-null  int64  
 3   NumberOfTime30-59DaysPastDueNotWorse  150000 non-null  int64  
 4   DebtRatio                             150000 non-null  float64
 5   MonthlyIncome                         120269 non-null  float64
 6   NumberOfOpenCreditLinesAndLoans       150000 non-null  int64  
 7   NumberOfTimes90DaysLate               150000 non-null  int64  
 8   NumberRealEstateLoansOrLines          150000 non-null  int64  
 9   NumberOfTime60-89DaysPastDueNotWorse  150000 non-null  int64  
 10  NumberOfDependents                    146076 

Se hace limpieza del dataset.

In [16]:
# --- LIMPIEZA ---
df_clean = df.copy()

# Renombrar columnas a español (más claro en el portafolio)
df_clean.columns = [
    'mora_target',             # Variable objetivo: 1 = entró en mora
    'util_lineas_rotativas',   # % uso de líneas de crédito
    'edad',
    'veces_mora_30_59',
    'ratio_deuda',
    'ingreso_mensual',
    'lineas_credito_abiertas',
    'veces_mora_90_dias',
    'inmuebles_hipotecados',
    'veces_mora_60_89',
    'dependientes'
]

# Imputar valores nulos con mediana (método seguro para principiantes)
df_clean['ingreso_mensual'].fillna(df_clean['ingreso_mensual'].median(), inplace=True)
df_clean['dependientes'].fillna(0, inplace=True)

# Eliminar outliers extremos en ingreso
df_clean = df_clean[df_clean['ingreso_mensual'] < 500000]
df_clean = df_clean[df_clean['edad'] > 18]

print("Dataset limpio:", df_clean.shape)
print("Nulos restantes:", df_clean.isnull().sum().sum())

Dataset limpio: (149987, 11)
Nulos restantes: 0


Se crea las nuevas variables: ratio deuda ingreso, severidad de mora e ingeso por dependiente.

In [17]:
# --- CREACIÓN DE VARIABLES NUEVAS ---
# Aquí está el valor analítico del proyecto

df_clean['ratio_deuda_ingreso'] = df_clean['ratio_deuda'] / (df_clean['ingreso_mensual'] + 1)

df_clean['frecuencia_mora'] = (
    df_clean['veces_mora_30_59'] +
    df_clean['veces_mora_60_89'] +
    df_clean['veces_mora_90_dias']
)

df_clean['severidad_mora'] = (
    df_clean['veces_mora_30_59'] * 1 +
    df_clean['veces_mora_60_89'] * 2 +
    df_clean['veces_mora_90_dias'] * 3
)

df_clean['ingreso_por_dependiente'] = df_clean['ingreso_mensual'] / (df_clean['dependientes'] + 1)

print("Nuevas variables creadas:")
print(df_clean[['ratio_deuda_ingreso', 'frecuencia_mora', 'severidad_mora']].describe())

Nuevas variables creadas:
       ratio_deuda_ingreso  frecuencia_mora  severidad_mora
count        149987.000000    149987.000000   149987.000000
mean             19.137387         0.927447        1.699841
std             347.803372        12.466743       24.929568
min               0.000000         0.000000        0.000000
25%               0.000025         0.000000        0.000000
50%               0.000065         0.000000        0.000000
75%               0.000289         0.000000        0.000000
max           60212.000000       294.000000      588.000000


Se usará el modelo RandomForest, por lo cual se procede a preparar los datos y features.

In [18]:
# --- PREPARAR DATOS PARA EL MODELO ---
features = [
    'util_lineas_rotativas', 'edad', 'ratio_deuda', 'ingreso_mensual',
    'lineas_credito_abiertas', 'dependientes',
    'ratio_deuda_ingreso', 'frecuencia_mora', 'severidad_mora',
    'ingreso_por_dependiente'
]

X = df_clean[features]
y = df_clean['mora_target']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# --- ENTRENAR MODELO ---
modelo = RandomForestClassifier(
    n_estimators=100,
    max_depth=8,
    random_state=42,
    class_weight='balanced'  # importante: datos desbalanceados
)

modelo.fit(X_train, y_train)
print("Modelo entrenado correctamente ✓")

Modelo entrenado correctamente ✓


Una vez entrendao el modelo con la data, se procede a evaluar escenario distintos y la importancia de las varaibles.

In [19]:
# --- EVALUAR MODELO ---
y_pred = modelo.predict(X_test)
y_proba = modelo.predict_proba(X_test)[:, 1]

print("=== REPORTE DE CLASIFICACIÓN ===")
print(classification_report(y_test, y_pred))

auc = roc_auc_score(y_test, y_proba)
print(f"\nROC-AUC Score: {auc:.4f}")
print("(Un valor > 0.70 es bueno para cobranza)")

# Importancia de variables
importancia = pd.DataFrame({
    'variable': features,
    'importancia': modelo.feature_importances_
}).sort_values('importancia', ascending=False)

fig = px.bar(importancia, x='importancia', y='variable',
             orientation='h', title='Variables más importantes para predecir mora',
             color='importancia', color_continuous_scale='Blues')
fig.show()

=== REPORTE DE CLASIFICACIÓN ===
              precision    recall  f1-score   support

           0       0.98      0.82      0.89     27993
           1       0.23      0.75      0.35      2005

    accuracy                           0.82     29998
   macro avg       0.61      0.79      0.62     29998
weighted avg       0.93      0.82      0.86     29998


ROC-AUC Score: 0.8676
(Un valor > 0.70 es bueno para cobranza)


Ahora se proced a una Segmentación Estratégica; entre alta, media y baja probabilidad de pago.

In [20]:
# --- SEGMENTACIÓN ESTRATÉGICA ---
df_resultado = X_test.copy()
df_resultado['prob_mora'] = y_proba
df_resultado['prob_pago'] = 1 - y_proba

# Clasificar en segmentos de cobranza
def segmentar(prob_pago):
    if prob_pago >= 0.75:
        return 'Alta probabilidad de pago'
    elif prob_pago >= 0.45:
        return 'Probabilidad media'
    else:
        return 'Baja probabilidad de pago'

def accion_recomendada(prob_pago):
    if prob_pago >= 0.75:
        return 'Llamada inmediata'
    elif prob_pago >= 0.45:
        return 'Enviar email / SMS'
    else:
        return 'Derivar a cobranza especializada'

df_resultado['segmento'] = df_resultado['prob_pago'].apply(segmentar)
df_resultado['accion'] = df_resultado['prob_pago'].apply(accion_recomendada)
df_resultado['prob_pago_%'] = (df_resultado['prob_pago'] * 100).round(1)

print("=== DISTRIBUCIÓN DE SEGMENTOS ===")
print(df_resultado['segmento'].value_counts())

=== DISTRIBUCIÓN DE SEGMENTOS ===
segmento
Alta probabilidad de pago    16340
Probabilidad media            8347
Baja probabilidad de pago     5311
Name: count, dtype: int64


A continuación se plasmará los resultados obtenidos en un dashboard final.

In [21]:
# --- DASHBOARD FINAL ---

# Tabla ranking de clientes (top 20)
top_clientes = df_resultado.sort_values('prob_pago', ascending=False).head(20).reset_index()
top_clientes['cliente_id'] = ['CLI-' + str(i).zfill(4) for i in top_clientes.index]

fig_tabla = go.Figure(data=[go.Table(
    header=dict(
        values=['Cliente ID', 'Prob. Pago %', 'Segmento', 'Acción Recomendada'],
        fill_color='#1f77b4',
        font=dict(color='white', size=12),
        align='center'
    ),
    cells=dict(
        values=[
            top_clientes['cliente_id'],
            top_clientes['prob_pago_%'],
            top_clientes['segmento'],
            top_clientes['accion']
        ],
        fill_color=[['#f0f8ff' if i % 2 == 0 else 'white'
                     for i in range(len(top_clientes))]],
        align='center'
    )
)])
fig_tabla.update_layout(title='🏦 Dashboard de Cobranza Inteligente — Top 20 clientes')
fig_tabla.show()

# Distribución de segmentos
fig_donut = px.pie(
    df_resultado,
    names='segmento',
    title='Distribución de clientes por segmento de cobranza',
    hole=0.4,
    color_discrete_map={
        'Alta probabilidad de pago': '#2ecc71',
        'Probabilidad media': '#f39c12',
        'Baja probabilidad de pago': '#e74c3c'
    }
)
fig_donut.show()

# Histograma de probabilidades
fig_hist = px.histogram(
    df_resultado, x='prob_pago_%',
    nbins=30,
    title='Distribución de probabilidad de pago',
    labels={'prob_pago_%': 'Probabilidad de pago (%)'},
    color_discrete_sequence=['#3498db']
)
fig_hist.show()

In [22]:
import joblib
joblib.dump(modelo, 'modelo_cobranza.pkl')
joblib.dump(features, 'features.pkl')

['features.pkl']